In [61]:
import pandas as pd

In [62]:
file_path = r"C:\Users\sneha\RFM-ANALYSIS\online_retail_II.csv"


In [63]:
df = pd.read_csv(file_path, encoding = "latin1")

In [64]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [65]:
df.shape


(1067371, 8)

In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [67]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [68]:
df =df.rename(columns= {"Customer ID":"Customer_ID"})

In [69]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [70]:
c_df =df[(df["Customer_ID"].notna()) &
        (df["Quantity"]>0) & 
        (df["Price"]>0) &
        (~df["Invoice"].astype(str).str.startswith("C"))].copy()

In [71]:
c_df.shape

(805549, 8)

# creating sales column

In [72]:
c_df["sales"] = c_df["Quantity"] * c_df["Price"]

In [73]:
c_df[["Customer_ID", "Invoice", "Quantity", "Price","sales"]].head(10)

,Customer_ID,Invoice,Quantity,Price,sales
0,13085.0,489434,12,6.95,83.4
1,13085.0,489434,12,6.75,81.0
2,13085.0,489434,12,6.75,81.0
3,13085.0,489434,48,2.10,100.8
4,13085.0,489434,24,1.25,30.0
5,13085.0,489434,24,1.65,39.6
6,13085.0,489434,24,1.25,30.0
7,13085.0,489434,10,5.95,59.5
8,13085.0,489435,12,2.55,30.6
9,13085.0,489435,12,3.75,45.0


In [74]:
c_df["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

In [75]:
snap_date = c_df["InvoiceDate"].max() + pd.Timedelta(days=1)

In [76]:
snap_date

Timestamp('2011-12-10 12:50:00')

In [77]:
last_purchase = c_df.groupby("Customer_ID")["InvoiceDate"].max()

# This finds the latest purchase date for every customer.Groups all rows belonging to the same customer.

In [78]:
last_purchase.head()

Customer_ID
12346.0   2011-01-18 10:01:00
12347.0   2011-12-07 15:52:00
12348.0   2011-09-25 13:13:00
12349.0   2011-11-21 09:51:00
12350.0   2011-02-02 16:01:00
Name: InvoiceDate, dtype: datetime64[us]

In [79]:
recency = (snap_date - last_purchase).dt.days

recency.head()

Customer_ID
12346.0    326
12347.0      2
12348.0     75
12349.0     19
12350.0    310
Name: InvoiceDate, dtype: int64

 # A lower Recency value is better because the customer bought more recently.
 # .dt.days give number of days.
 # before .dt.days----> 5 days 00:00:00,after---> 5

# Above all this about R IN RFM

In [80]:
frequency = c_df.groupby("Customer_ID")["Invoice"].nunique()

frequency.head()

Customer_ID
12346.0    12
12347.0     8
12348.0     5
12349.0     4
12350.0     1
Name: Invoice, dtype: int64

nunique() --> count unique values
We count unique invoices because one invoice can contain many product rows, but it is still one order.

In [81]:
monetary = c_df.groupby("Customer_ID")["sales"].sum()

monetary.head()


Customer_ID
12346.0    77556.46
12347.0     5633.32
12348.0     2019.40
12349.0     4428.69
12350.0      334.40
Name: sales, dtype: float64

In [82]:
example_customer = c_df["Customer_ID"].iloc[0]

c_df[c_df["Customer_ID"] == example_customer][
    ["Customer_ID", "Invoice", "InvoiceDate", "sales"]
].head(20)

,Customer_ID,Invoice,InvoiceDate,sales
0,13085.0,489434,2009-12-01 07:45:00,83.4
1,13085.0,489434,2009-12-01 07:45:00,81.0
2,13085.0,489434,2009-12-01 07:45:00,81.0
3,13085.0,489434,2009-12-01 07:45:00,100.8
4,13085.0,489434,2009-12-01 07:45:00,30.0
5,13085.0,489434,2009-12-01 07:45:00,39.6
6,13085.0,489434,2009-12-01 07:45:00,30.0
7,13085.0,489434,2009-12-01 07:45:00,59.5
8,13085.0,489435,2009-12-01 07:46:00,30.6
9,13085.0,489435,2009-12-01 07:46:00,45.0


In [83]:
monetary.loc[example_customer]

np.float64(2433.28)

In [84]:
rfm = pd.DataFrame({
    "Recency": recency,
    "Frequency": frequency,
    "Monetary": monetary
})

rfm.head()

,Recency,Frequency,Monetary
Customer_ID,,,
12346.0,326,12,77556.46
12347.0,2,8,5633.32
12348.0,75,5,2019.40
12349.0,19,4,4428.69
12350.0,310,1,334.40


Table name     What it contains 

df             Original raw dataset 
c_df           Cleaned transaction dataset 
recency         Recency values only 
frequency        Frequency values only
monetary        Monetary values only
rfm              All three RFM values together
rfm is the customer-level table we will use for segmentation.It has one row per customer, instead of many product rows per customer.

In [85]:
rfm = rfm.reset_index()

rfm.head()

,Customer_ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


# customer_id a normal visible column.
Before this, customer_id is a hidden row label called an index.

In [86]:
rfm[["Recency", "Frequency", "Monetary"]].describe()

,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000
mean,201.331916,6.289384,3018.616737
std,209.338707,13.009406,14737.731040
min,1.000000,1.000000,2.950000
25%,26.000000,1.000000,348.762500
50%,96.000000,3.000000,898.915000
75%,380.000000,7.000000,2307.090000
max,739.000000,398.000000,608821.650000


count — number of customers
mean — average value
min — smallest value
max — largest value
25%, 50%, 75% — customer value ranges

before giving RFM scores from 1 to 5

In [87]:
recency_rank = rfm["Recency"].rank(method="first")

recency_rank.head()
recency_rank.head().max()

recency_rank.head().min()

np.float64(94.0)

Sometimes two customers have the same Recency
method="first" gives them different ranks based on which one appears first in the table

In [88]:
recency_groups = pd.qcut(recency_rank, q=5)

recency_groups.head()

0    (3527.2, 4702.6]
1     (0.999, 1176.4]
2    (2351.8, 3527.2]
3     (0.999, 1176.4]
4    (3527.2, 4702.6]
Name: Recency, dtype: category
Categories (5, interval[float64, right]): [(0.999, 1176.4] < (1176.4, 2351.8] < (2351.8, 3527.2] < (3527.2, 4702.6] < (4702.6, 5878.0]]


(  = do not include the left value
]  = include the right value
Pandas uses 0.999 instead of 1 so the first ranked customer, with rank 1, is included.
The decimal 1189.2 is only a mathematical boundary pandas creates while splitting customers into equal groups

pd.qcut() divides values into equal-sized groups

In [89]:
rfm["R_score"] = pd.qcut(recency_rank,q=5,labels=[5, 4, 3, 2, 1])

In [90]:
rfm[["Recency", "R_score"]].head()

,Recency,R_score
0,326,2
1,2,5
2,75,3
3,19,5
4,310,2


The first group—customers with the lowest Recency days—gets score 5.
The last group—customers with the highest Recency days—gets score 1

In [91]:
frequency_rank = rfm["Frequency"].rank(method="first")

frequency_rank.head()

0    5134.0
1    4580.0
2    3718.0
3    3232.0
4       1.0
Name: Frequency, dtype: float64

Few orders  - low rank
Many orders - high rank
Unlike Recency, higher Frequency is better.

In [92]:
rfm["F_score"] = pd.qcut(frequency_rank,q=5,labels=[1, 2, 3, 4, 5]).astype(int)

In [93]:
rfm[["Frequency", "F_score"]].head()

,Frequency,F_score
0,12,5
1,8,4
2,5,4
3,4,3
4,1,1


In [94]:
monetary_rank = rfm["Monetary"].rank(method="first")

monetary_rank.head()

0    5860.0
1    5300.0
2    4232.0
3    5112.0
4    1409.0
Name: Monetary, dtype: float64

Low total spending  - low rank
High total spending - high rank

In [96]:
rfm["M_score"] = pd.qcut(monetary_rank,q=5,labels=[1, 2, 3, 4, 5]).astype(int)

In [97]:
rfm[["Monetary", "M_score"]].head()

,Monetary,M_score
0,77556.46,5
1,5633.32,5
2,2019.40,4
3,4428.69,5
4,334.40,2


In [100]:
rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)

 if error comes -- Why this happened: pd.qcut() creates groups called categories. They look like numbers (1, 2, 3, 4, 5), but pandas still treats them as labels. astype(int) changes those labels into real numeric values so Python can add them.

In [101]:
rfm["RFM_score"] = (
    rfm["R_score"] +
    rfm["F_score"] +
    rfm["M_score"]
)

rfm[["Recency", "Frequency", "Monetary", "R_score", "F_score", "M_score", "RFM_score"]].head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score
0,326,12,77556.46,2,5,5,12
1,2,8,5633.32,5,4,5,14
2,75,5,2019.40,3,4,4,11
3,19,4,4428.69,5,3,5,13
4,310,1,334.40,2,1,2,5


# creating the customer segments

In [106]:
def customer_segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champion"
    
    elif row["R_score"] >= 3 and row["F_score"] >= 3:
        return "Loyal"
    
    elif row["R_score"] <= 2 and (
        row["F_score"] >= 3 or row["M_score"] >= 3
    ):
        return "At Risk"
    
    elif row["R_score"] == 1 and row["F_score"] <= 2 and row["M_score"] <= 2:
        return "Lost"
    
    else:
        return "Potential Loyalist"

Champion: purchased recently, orders often, spends a lot.
Loyal: reasonably recent and buys repeatedly.
At Risk: has not purchased recently, but had reasonable frequency or spending.
Lost: inactive, infrequent, low spending.
Other: customers who do not fit these four groups exactly.

In [107]:
rfm["Segment"] = rfm.apply(customer_segment, axis=1)

rfm[["Customer_ID", "R_score", "F_score", "M_score", "RFM_score", "Segment"]].head(10)

,Customer_ID,R_score,F_score,M_score,RFM_score,Segment
0,12346.0,2,5,5,12,At Risk
1,12347.0,5,4,5,14,Champion
2,12348.0,3,4,4,11,Loyal
3,12349.0,5,3,5,13,Loyal
4,12350.0,2,1,2,5,Potential Loyalist
5,12351.0,2,1,2,5,Potential Loyalist
6,12352.0,4,5,4,13,Champion
7,12353.0,2,2,2,6,Potential Loyalist
8,12354.0,2,1,3,6,At Risk
9,12355.0,2,2,3,7,At Risk


In [108]:
rfm["Segment"].value_counts()

Segment
Loyal                 1410
Potential Loyalist    1323
Champion              1292
At Risk               1074
Lost                   779
Name: count, dtype: int64

#creating a summary table for each segment

In [110]:
segment_summary = rfm.groupby("Segment").agg(
    Customers=("Customer_ID", "count"),
    Total_Revenue=("Monetary", "sum"),
    Average_Spend=("Monetary", "mean"),
    Average_Recency=("Recency", "mean"),
    Average_Orders=("Frequency", "mean")
).round(2)

segment_summary

,Customers,Total_Revenue,Average_Spend,Average_Recency,Average_Orders
Segment,,,,,
At Risk,1074,1975114.27,1839.03,379.49,4.17
Champion,1292,12087432.74,9355.60,19.73,17.13
Lost,779,192051.12,246.54,562.02,1.15
Loyal,1410,2753643.30,1952.94,71.24,5.45
Potential Loyalist,1323,735187.74,555.70,160.33,1.35


This table is important because it tells us which segments have the most customers, bring the most revenue, spend more on average, and have been inactive longer.

In [111]:
segment_summary = segment_summary.sort_values( by="Total_Revenue",ascending=False)

segment_summary

,Customers,Total_Revenue,Average_Spend,Average_Recency,Average_Orders
Segment,,,,,
Champion,1292,12087432.74,9355.60,19.73,17.13
Loyal,1410,2753643.30,1952.94,71.24,5.45
At Risk,1074,1975114.27,1839.03,379.49,4.17
Potential Loyalist,1323,735187.74,555.70,160.33,1.35
Lost,779,192051.12,246.54,562.02,1.15


above helps answers ,Which customer segment brings the most money to the business?

In [112]:
segment_summary["Revenue_Share_%"] = (
    segment_summary["Total_Revenue"] /
    segment_summary["Total_Revenue"].sum() * 100
).round(2)

One segment, for example At Risk, brought £200,000.
All customer segments together brought £1,000,000.
So At Risk customers produced 20% of total revenue.

In [113]:
segment_summary

,Customers,Total_Revenue,Average_Spend,Average_Recency,Average_Orders,Revenue_Share_%
Segment,,,,,,
Champion,1292,12087432.74,9355.60,19.73,17.13,68.12
Loyal,1410,2753643.30,1952.94,71.24,5.45,15.52
At Risk,1074,1975114.27,1839.03,379.49,4.17,11.13
Potential Loyalist,1323,735187.74,555.70,160.33,1.35,4.14
Lost,779,192051.12,246.54,562.02,1.15,1.08


In [115]:
rfm.to_csv("customer_segments_using_rfm.csv", index=False)

In [114]:
segment_summary.to_csv("segment_summary_rfm.csv")